In [7]:
!pip install endee sentence-transformers pypdf2 langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 20.3 MB/s eta 0:00:00


In [44]:
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from PyPDF2 import PdfReader
import textwrap
from google.colab import files
import json

In [10]:
model_name = "BAAI/bge-m3"

In [11]:
embedder = SentenceTransformer(model_name, device="cuda")
print(f"Loaded {model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Loaded BAAI/bge-m3


In [14]:
uploaded = files.upload()

Saving The-Gale-Encyclopedia-of-Medicine-3rd-Edition-staibabussalamsula.ac_.id_.pdf to The-Gale-Encyclopedia-of-Medicine-3rd-Edition-staibabussalamsula.ac_.id_.pdf


In [15]:
pdf_filename = next(iter(uploaded))
print(f"Uploaded file: {pdf_filename}")

Uploaded file: The-Gale-Encyclopedia-of-Medicine-3rd-Edition-staibabussalamsula.ac_.id_.pdf


In [19]:
reader = PdfReader(pdf_filename)

In [21]:
full_text = ""

for page_num, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    full_text += text + "\n\n"
    print(f"Page {page_num} extracted ({len(text)} chars)")

Page 1 extracted (0 chars)
Page 2 extracted (46 chars)
Page 3 extracted (96 chars)
Page 4 extracted (96 chars)
Page 5 extracted (96 chars)
Page 6 extracted (96 chars)
Page 7 extracted (124 chars)
Page 8 extracted (3627 chars)
Page 9 extracted (718 chars)
Page 10 extracted (2090 chars)
Page 11 extracted (2835 chars)
Page 12 extracted (2758 chars)
Page 13 extracted (2693 chars)
Page 14 extracted (2669 chars)
Page 15 extracted (2855 chars)
Page 16 extracted (2442 chars)
Page 17 extracted (2633 chars)
Page 18 extracted (2882 chars)
Page 19 extracted (2749 chars)
Page 20 extracted (2640 chars)
Page 21 extracted (2399 chars)
Page 22 extracted (1186 chars)
Page 23 extracted (3113 chars)
Page 24 extracted (1053 chars)
Page 25 extracted (577 chars)
Page 26 extracted (2037 chars)
Page 27 extracted (2452 chars)
Page 28 extracted (2491 chars)
Page 29 extracted (2515 chars)
Page 30 extracted (2146 chars)
Page 31 extracted (3306 chars)
Page 32 extracted (5090 chars)
Page 33 extracted (4494 chars)
Pa

In [22]:
print(f"Total extracted text length: {len(full_text):} characters")

Total extracted text length: 17913513 characters


In [23]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=120,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
    keep_separator=True
)

In [24]:
chunks = text_splitter.split_text(full_text.strip())

In [25]:
embeddings = embedder.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

In [26]:
embeddings = np.array(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

Embeddings shape: (39991, 1024)


In [45]:
np.save("pdf_embeddings.npy", embeddings)

In [46]:
metadata = []
for i, chunk in enumerate(chunks):
    metadata.append({
        "id": f"chunk_{i+1}",
        "text": chunk,
        "source": pdf_filename,
        "chunk_index": i,
        "length": len(chunk)
    })

In [47]:
with open("pdf_chunks_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)